# Spirula → bizim eğitici · Colab TEST v1
**Amaç:** Spirula'nın hazır görüntülerini, kamera pozlarını ve başlangıç noktalarını kullanarak kendi statik eğiticimizi büyük GPU'da denemek. **COLMAP/SfM, frame extraction, SAM, optical flow veya depth modelleri yeniden çalıştırılmaz.**

Bu bağımsız test notebook'u mevcut projedeki eğitim kodunun sabit bir kopyasını içerir; GitHub token'ı ya da repo klonlamak gerekmez. Üretim notebook oluşturucusu değiştirilmedi.

## Önce bilgisayarında
1. Videoyu Spirula'da dataset'e dönüştür. Normal telefon videosu kullan; tek lensle çek, zoom/lens değiştirme. Bu test tek kamera kalibrasyonunu destekler; 360/fisheye ve birden fazla kamera kalibrasyonunda anlaşılır hatayla durur.
2. Kamera yerleşimini kontrol et. Birbirinden kopuk modelleri tek odaymış gibi eğitme. Notebook modellerin görüntü sayılarını listeler; `MODEL_NAME` ile doğru parçayı seçersin.
3. Dataset klasöründen **`images/` ve `sparse/` klasörlerini** aynı üst klasör altında ZIP yap. Dosya adlarını/alt klasörleri değiştirme.

```text
myroom_spirula.zip
└── myroom_spirula_dataset/    # Bu üst klasör olabilir veya olmayabilir
    ├── images/               # Tüm fotoğraflar, özgün isimleriyle
    └── sparse/
        └── 0/
            ├── cameras.bin
            ├── images.bin
            └── points3D.bin
```

4. ZIP'i Drive → **MyDrive → GaussianTests → inputs** içine yükle. Alternatif: dataset klasörünü aynı yapıyla yükleyebilirsin; ZIP çok sayıda küçük dosyada daha pratiktir.
5. **Video, `outputs/`, `features/`, `matches.bin`, `.progress/`, `.resume/` yüklemene gerek yok.** `normals/`, `depths/`, `masks/` bu v1 testinde kullanılmaz. Spirula'nın normal/renk düzeltme özellikleri bizim eğiticide bulunmadığından aynı kalite garantisi yoktur.
6. Colab'da **Runtime / Çalışma zamanı → Change runtime type → GPU** seç. `quality_3m` için en az 38 GiB, `stress_5m` için en az 70 GiB GPU belleği gerekir; uygun GPU verilmediyse kod durur. Yüksek sistem RAM'i de seç.
7. Hücreleri yukarıdan aşağı çalıştır. İlk GPU kernel derlemesi birkaç dakika sürebilir; bu veri hazırlığı değildir.

**İlk koşu önerisi:** `quality_3m` → 50.000 adım, 3 milyon nominal splat sınırı, en uzun kenar 1920. Fazla splat daha iyi kalite garantisi değildir. `stress_5m` ikinci deney içindir.

**Bilinen sınır:** Mevcut eğiticinin splat sınırı sert değildir, bir büyüme adımında aşılabilir. Gerçek sayı raporlanır. Eğitim ara kayıtları optimizer durumunu içermediğinden bu notebook kesintiden tam eğitim devamı sunmaz; ara kayıttan PLY kurtarma hücresi vardır.


In [ ]:
# 1 — Drive ve senin ayarların
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/GaussianTests/inputs/myroom_spirula.zip" # @param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/GaussianTests/results" # @param {type:"string"}
SCENE_NAME = "myroom" # @param {type:"string"}
MODEL_NAME = "0" # @param {type:"string"}
PRESET = "quality_3m" # @param ["baseline_1m", "quality_3m", "stress_5m"]
SEED = 42

# Her çalıştırmada ayrı klasör: mevcut sonuçların üzerine yazmaz.
from pathlib import Path
import datetime, uuid, re
assert re.fullmatch(r'[A-Za-z0-9_-]+', SCENE_NAME), 'Sahne adında yalnızca harf/rakam/_/- kullan.'
assert MODEL_NAME.isdigit(), 'MODEL_NAME sparse altındaki sayısal klasör adı olmalı.'
RUN_ID = SCENE_NAME + '_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/spirula_tests') / RUN_ID
DRIVE_OUT = Path(OUTPUT_ROOT) / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
DRIVE_OUT.mkdir(parents=True, exist_ok=False)
print('Drive sonuç klasörü:', DRIVE_OUT)


In [ ]:
# 2 — Yalnızca eğitim bağımlılıkları. Colab'ın kurulu PyTorch'unu değiştirmez.
import subprocess, sys, os
os.environ['MAX_JOBS'] = '4'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'gsplat==1.5.3', 'pytorch-msssim==1.0.0', 'plyfile', 'ninja',
               'opencv-python-headless', 'psutil'], check=True)
import torch, numpy as np, cv2, psutil
assert torch.cuda.is_available(), 'GPU seçilmemiş. Runtime → Change runtime type → GPU.'
PROFILES = {
    'baseline_1m': dict(steps=30000, cap=1000000, long_edge=1920, min_gib=20),
    'quality_3m': dict(steps=50000, cap=3000000, long_edge=1920, min_gib=38),
    'stress_5m': dict(steps=60000, cap=5000000, long_edge=2560, min_gib=70),
}
CFG = dict(PROFILES[PRESET])
CFG.update(steps=ITERATIONS_OVERRIDE, cap=GAUSSIANS_OVERRIDE, min_gib=MIN_VRAM_OVERRIDE)
gpu = torch.cuda.get_device_properties(0)
gib = gpu.total_memory / 2**30
print(f'GPU: {gpu.name} | VRAM: {gib:.1f} GiB | RAM: {psutil.virtual_memory().available/2**30:.1f} GiB')
print('Preset:', PRESET, CFG)
assert gib >= CFG['min_gib'], f"Bu preset en az {CFG['min_gib']} GiB VRAM ister; daha küçük preset veya daha büyük GPU seç."
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)


## Eğitim kodu
Notebook oluşturulduğu anda projedeki trainer kodu SHA-256 doğrulamasıyla paketlenir. Harici yerel dosya gerekmez.

In [ ]:
import base64, hashlib, io, zipfile
BUNDLE_SHA256 = ""
BUNDLE_B64 = ""

raw = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256
SOURCE = WORK / 'source'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        assert (SOURCE / name).resolve().is_relative_to(SOURCE.resolve())
    z.extractall(SOURCE)
sys.path.insert(0, str(SOURCE))
from spirula_import import convert_dataset
from backend.preprocess.parse_colmap import parse_cameras, load_points3d, scene_extent
from backend.model.gaussian_model import GaussianModel
from backend.model.deformation import DeformationField
from backend.model.trainer import Trainer4DGS
from backend.model.renderer import render_view
from backend.export.to_splat import export_to_ply, write_ply
print('Eğitim kodu hazır; kaynak özeti:', BUNDLE_SHA256)

In [ ]:
# 4 — Dataset'i Drive'dan YEREL diske al, ZIP ise güvenli biçimde aç.
import shutil, stat, time, json, struct
source = Path(DATASET_PATH)
assert source.exists(), f'Yol bulunamadı: {source}'
t0 = time.perf_counter()
staging = WORK / 'input'
if source.is_dir():
    assert (source/'images').is_dir() and (source/'sparse').is_dir(), 'Dataset kök klasörünü seç.'
    size = sum(p.stat().st_size for sub in ('images','sparse') for p in (source/sub).rglob('*') if p.is_file())
    assert shutil.disk_usage(WORK).free > size * 2 + 15*2**30, 'Yerel disk alanı yetersiz.'
    staging.mkdir()
    for sub in ('images','sparse'): shutil.copytree(source/sub, staging/sub)
else:
    assert source.suffix.lower() == '.zip', 'ZIP veya dataset klasörü seç.'
    local_zip = WORK / 'dataset.zip'
    assert shutil.disk_usage(WORK).free > source.stat().st_size + 15*2**30, 'ZIP için yerel disk alanı yetersiz.'
    shutil.copyfile(source, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        size = sum(i.file_size for i in z.infolist())
        assert shutil.disk_usage(WORK).free > size * 2 + 15*2**30, 'Açılmış dataset için disk alanı yetersiz.'
        for i in z.infolist():
            dest = (staging / i.filename).resolve()
            assert dest.is_relative_to(staging.resolve()), 'Geçersiz ZIP yolu.'
            assert not stat.S_ISLNK(i.external_attr >> 16), 'ZIP içinde sembolik bağlantı desteklenmiyor.'
        z.extractall(staging)
candidates = ([staging] if (staging/'images').is_dir() and (staging/'sparse').is_dir() else [])
candidates += [p.parent for p in staging.rglob('images') if p.is_dir() and (p.parent/'sparse').is_dir()]
candidates = list(dict.fromkeys(candidates))
assert len(candidates) == 1, f'ZIP içinde tek dataset bekleniyor; bulunan: {candidates}'
DATASET = candidates[0]
print('Yerel dataset:', DATASET)
for m in sorted((DATASET/'sparse').iterdir()):
    if (m/'images.bin').is_file():
        with (m/'images.bin').open('rb') as f: n=struct.unpack('<Q', f.read(8))[0]
        print(f'Model {m.name}: {n} hizalanmış görüntü' + (' ← seçili' if m.name==MODEL_NAME else ''))
print(f'Drive aktarımı/açma: {time.perf_counter()-t0:.1f} saniye')


## 5 — Kamera uyarlama (yeniden SfM değil)
OpenCV lens bozulması görüntüler ve kamera birlikte düzeltilerek pinhole modele çevrilir. Bu küçük görüş alanı değişimi oluşturabilir. Fotoğraf/poz eşleşmeleri korunur. Görüntüler preset çözünürlüğüne indirilir; büyütülmez. Bu hücre hata verirse eğitim hücresine geçme.


In [ ]:
PREPARED = WORK / 'prepared'
report = convert_dataset(DATASET, PREPARED, MODEL_NAME, CFG['long_edge'])
cams = parse_cameras(PREPARED/'colmap')
xyz, rgb = load_points3d(PREPARED/'colmap')
names = sorted(cams)
assert len(names) == report['images']
assert all((PREPARED/'frames'/n).is_file() for n in names)
assert np.isfinite(xyz).all()
# Eğiticide tek K kullanılır; farklı kameraları sessizce yanlış okumayalım.
assert all(np.allclose(c['K'], cams[names[0]]['K']) for c in cams.values())
W,H = report['width'], report['height']
image_cache_bytes = len(names)*W*H*3
available = psutil.virtual_memory().available
print(f'{len(names)} görüntü, {len(xyz):,} nokta; {W}×{H}; dönüşüm {report["seconds"]:.1f} saniye')
print(f'Tahmini RGB cache: {image_cache_bytes/2**30:.1f} GiB; boş RAM: {available/2**30:.1f} GiB')
assert image_cache_bytes < available*0.50, 'Görüntü cache fazla: yüksek RAM seç veya daha az görüntülü dataset/düşük çözünürlük kullan. Eğitim başlatılmadı.'
# Her 10. fotoğraf eğitimden ayrılır. SfM tüm görüntülerden geldi: bu tamamen bağımsız bir yakalama testi değildir.
HOLDOUT = list(range(0,len(names),10))
print(f'Eğitim: {len(names)-len(HOLDOUT)}; ayrılan değerlendirme: {len(HOLDOUT)}')
shutil.copy2(PREPARED/'import_report.json', DRIVE_OUT/'import_report.json')
manifest = dict(run_id=RUN_ID, preset=PRESET, config=CFG, seed=SEED, gpu=gpu.name,
                torch=torch.__version__, cuda=torch.version.cuda, source_sha256=BUNDLE_SHA256,
                registered_images=len(names), sparse_points=len(xyz), width=W, height=H,
                holdout_indices=HOLDOUT, nominal_cap_is_hard=False,
                normals=False, depth=False, exposure_correction=False, status='prepared')
(DRIVE_OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# 6 — GPU render + backward kontrolü. Başarısızsa uzun eğitim başlamaz.
t0=time.perf_counter()
m=torch.tensor([[0.,0.,2.]],device='cuda',requires_grad=True)
q=torch.tensor([[1.,0.,0.,0.]],device='cuda')
s=torch.full((1,3),0.1,device='cuda')
a=torch.tensor([0.5],device='cuda',requires_grad=True)
c=torch.full((1,3),0.5,device='cuda',requires_grad=True)
k=torch.tensor([[50.,0.,32.],[0.,50.,32.],[0.,0.,1.]],device='cuda')
r,_,_=render_view(m,q,s,a,c,k,torch.eye(4,device='cuda'),64,64,sh_degree=0)
r.sum().backward()
assert m.grad is not None and torch.isfinite(m.grad).all() and torch.isfinite(r).all()
torch.cuda.synchronize()
print(f'GPU render/backward OK ({time.perf_counter()-t0:.1f} s; ilk derleme dahil).')
del m,q,s,a,c,k,r
torch.cuda.empty_cache()


## Eğitimi başlat
Her 5.000 adımda model checkpoint kaydedilir. Optimizer durumu yoktur: yalnızca PLY kurtarma içindir, tam eğitim devamı değildir. Anlık PSNR bağımsız kalite skoru değildir. Gaussian bütçesi nominaldir; büyüme adımında aşılabilir.

In [ ]:
import gc, types
from IPython.display import display
from pytorch_msssim import ssim

# Aynı kaynak ve ayarlarla tekrarlanabilir başlangıç; GPU sonucu bit-bit eşit olmak zorunda değildir.
torch.manual_seed(SEED); np.random.seed(SEED)
t0=time.perf_counter()
# Çok büyük sparse başlangıcında pahalı CPU mesafe hesabını sınırlıyoruz; deterministik örnekleme.
init_ids=np.arange(len(xyz))
if len(init_ids)>150000:
    init_ids=np.sort(np.random.default_rng(SEED).choice(len(xyz),150000,replace=False))
gs=GaussianModel(torch.from_numpy(xyz[init_ids]), torch.from_numpy(rgb[init_ids].astype('float32')/255), fourier_K=0)
init_seconds=time.perf_counter()-t0
print(f'Başlangıç: {len(init_ids):,} splat; CPU init {init_seconds:.1f} s')
deform=DeformationField(resolution=8,feat_dim=4,mlp_width=32,mlp_depth=1,num_time_freqs=2)
for p in deform.parameters(): p.requires_grad_(False)
trainer=Trainer4DGS(gs,deform,scene_extent=scene_extent(xyz),max_gaussians=CFG['cap'],
    density_end_iter=min(25000,CFG['steps']//2),deform_pos_mode='mlp',
    lambda_deform_reg=0,lambda_smoothness=0,lambda_rigidity=0,lambda_fourier_reg=0,lambda_mask_motion=0)
# Periyodik checkpoint'in atomik Drive kopyası; optimizer state saklanmaz.
original_save=trainer._save_checkpoint
last_saved=None

def mirror_save(folder,it,final=False):
    global last_saved
    original_save(folder,it,final)
    filename='ckpt_final.pt' if final else f'ckpt_{it:06d}.pt'
    p=Path(folder)/filename
    tmp=DRIVE_OUT/'latest_checkpoint.pt.partial'
    shutil.copy2(p,tmp); os.replace(tmp,DRIVE_OUT/'latest_checkpoint.pt')
    (DRIVE_OUT/'checkpoint_info.json').write_text(json.dumps(dict(iter=it,final=final,exact_training_resume=False)))
    last_saved=it
    print(f'Drive ara kaydı: {it:,}')
trainer._save_checkpoint=mirror_save
started=time.perf_counter()
last_iter=0
metrics_path=WORK/'metrics.jsonl'
def progress(it,total,loss,psnr,n):
    global last_iter
    last_iter=it
    elapsed=time.perf_counter()-started
    record=dict(iter=it,total=total,loss=loss,training_frame_psnr=psnr,splats=n,elapsed_s=elapsed,
                allocated_gib=torch.cuda.memory_allocated()/2**30)
    with metrics_path.open('a') as f:f.write(json.dumps(record)+chr(10))
    tmp=DRIVE_OUT/'progress.json.partial';tmp.write_text(json.dumps(record));os.replace(tmp,DRIVE_OUT/'progress.json')
manifest.update(status='training',init_points=len(init_ids),init_seconds=init_seconds)
(DRIVE_OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
try:
    history=trainer.train([PREPARED/'frames'/n for n in names],
        torch.from_numpy(cams[names[0]]['K']).float(),
        [torch.from_numpy(cams[n]['w2c']).float() for n in names],
        n_iters=CFG['steps'],image_size=(W,H),static_mode=True,preload_to_ram=True,
        log_interval=100,ckpt_dir=WORK/'checkpoints',ckpt_interval=5000,
        sh_progressive_schedule=True,holdout_indices=HOLDOUT,progress_callback=progress)
    torch.cuda.synchronize()
    manifest.update(status='trained',training_seconds=time.perf_counter()-started,final_splats=gs.num_points)
except BaseException as e:
    manifest.update(status='interrupted_or_failed',last_logged_iter=last_iter,error=repr(e),last_drive_checkpoint=last_saved)
    raise
finally:
    (DRIVE_OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
    if metrics_path.exists():shutil.copy2(metrics_path,DRIVE_OUT/'metrics.jsonl')
print(f'Eğitim bitti: {manifest["training_seconds"]/60:.1f} dk; {gs.num_points:,} splat')


In [ ]:
# 8 — Final PLY'yi Drive'a kaydet (değerlendirmeden önce çıktıyı korur).
assert manifest['status']=='trained', 'Önce eğitim başarıyla tamamlanmalı.'
final_paths=export_to_ply(gs,None,WORK/'final',num_timestamps=1)
final_ply=DRIVE_OUT/'final_30000.ply' if CFG['steps']==30000 else DRIVE_OUT/f'final_{CFG["steps"]}.ply'
shutil.copy2(final_paths[0],str(final_ply)+'.partial');os.replace(str(final_ply)+'.partial',final_ply)
manifest['final_ply']=str(final_ply);manifest['status']='exported'
(DRIVE_OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('FINAL MODEL:', final_ply)
print(f'Dosya: {final_ply.stat().st_size/2**20:.1f} MiB')
# Eğitim optimizer'ını serbest bırak; model GPU'da değerlendirme için kalır.
trainer.optimizer.state.clear();gc.collect();torch.cuda.empty_cache()


In [ ]:
# 9 — Eğitimde kullanılmayan fotoğraflardan en fazla 16 tanesinde ölçüm + yan yana görüntüler.
from PIL import Image
EVAL_DIR=DRIVE_OUT/'evaluation';EVAL_DIR.mkdir(exist_ok=True)
eval_ids=[HOLDOUT[i] for i in np.unique(np.linspace(0,len(HOLDOUT)-1,min(16,len(HOLDOUT))).astype(int))]
results=[]
with torch.no_grad():
    for j,idx in enumerate(eval_ids):
        name=names[idx];gt_np=np.asarray(Image.open(PREPARED/'frames'/name).convert('RGB')).copy()
        gt=torch.from_numpy(gt_np).to('cuda',dtype=torch.float32)/255
        rendered,_,_=render_view(gs.means,gs.get_quats,gs.get_scales,gs.get_opacities,gs.get_colors,
            torch.from_numpy(cams[name]['K']).float().cuda(),torch.from_numpy(cams[name]['w2c']).float().cuda(),W,H,sh_degree=3)
        pred=rendered.clamp(0,1)
        mse=(pred-gt).square().mean().item()
        score=-10*np.log10(max(mse,1e-12))
        sv=ssim(pred.permute(2,0,1)[None],gt.permute(2,0,1)[None],data_range=1.).item()
        results.append(dict(frame=name,psnr=score,ssim=sv))
        rgb_pred=(pred.cpu().numpy()*255).astype('uint8')
        pair=Image.fromarray(np.concatenate([gt_np,rgb_pred],axis=1))
        pair.save(EVAL_DIR/f'{idx:06d}_input_left_render_right.jpg',quality=93)
        if j<3:
            preview=pair.copy();preview.thumbnail((1400,700));display(preview)
summary=dict(evaluated_views=len(results),mean_psnr=float(np.mean([x['psnr'] for x in results])),
             mean_ssim=float(np.mean([x['ssim'] for x in results])),views=results,
             note='Fotoğraflar eğitimden ayrıldı; SfM tüm görüntülerden geldi. En fazla 16 örnek, tam bağımsız yakalama testi değil.')
(EVAL_DIR/'metrics.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False))
manifest['status']='complete';(DRIVE_OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Örnek değerlendirme PSNR:',summary['mean_psnr'],'SSIM:',summary['mean_ssim'])
print('Bütün sonuçlar:',DRIVE_OUT)


## Sonuçları nasıl paylaşacaksın?
Drive'daki bu çalışmanın klasöründen **`final_*.ply`, `run_manifest.json`, `metrics.jsonl`, `import_report.json` ve `evaluation/`** klasörünü sakla. Bana sonuç klasörü yolunu ver. Fotoğraf/render çiftlerinde **sol gerçek fotoğraf, sağ bizim render**. Son model standard PLY olduğu için yan yana görüntüleyicimizde açabiliriz.

**Hata durumunda:** GPU bellek hatasında `baseline_1m` ile yeni çalışma başlat veya daha büyük GPU seç. RAM hatasında yüksek RAM/daha düşük çözünürlük gerekir. Bozuk kamera/model hatasını daha fazla adım veya splat ile çözmeye çalışma.

**Spirula ile adil kıyas:** aynı dataset, aynı kamera açıları, aynı çözünürlük ve benzer gerçek splat sayısı kullan. Bu testte fotoğrafların %10'u eğitimden ayrılıyor; Spirula tüm fotoğraflarla eğitildiyse bunu karşılaştırma notuna yaz.

### Opsiyonel — Kesintiden sonra ara model kurtarma
Aşağıdaki hücreyi yalnızca eğitim kesildiyse çalıştır. Yeni oturumdaysan önce 1–3. hücreleri çalıştır; sonra eski sonuç klasöründeki checkpoint yolunu gir. **Optimizer olmadığı için eğitime tam kaldığı yerden devam etmez; yalnızca PLY çıkarır.**


In [ ]:
# 10 — OPSİYONEL: yalnızca kendi ürettiğin güvenilir checkpoint'i kullan.
RECOVERY_CHECKPOINT = "" # @param {type:"string"}
if RECOVERY_CHECKPOINT:
    p=Path(RECOVERY_CHECKPOINT)
    ck=torch.load(p,map_location='cpu',weights_only=True)
    state=ck['gs']
    def arr(k):return state[k].detach().cpu().float().numpy()
    q=arr('quats');q=q/np.maximum(np.linalg.norm(q,axis=1,keepdims=True),1e-8)
    recovered=p.parent/f'recovered_{ck["iter"]}.ply'
    write_ply(arr('means'),arr('scales'),q,arr('opacities').reshape(-1),arr('sh_dc').reshape(-1,3),arr('sh_rest'),recovered)
    print('Kurtarılan ara model:',recovered)
else:
    print('Kurtarma atlandı; normal tamamlanan koşuda gerekli değil.')
